#### Final lookup: what is actually going into training

`pack_dataset.ipynb` writes `train.bin`, `val_en.bin`, `val_hi.bin`, `val_mr.bin` to `dataset/final_packed/`. Those are the literal model inputs: flat `uint16` files, one 2048 token row after another, nothing else needed before training starts. This notebook decodes a handful of real rows back into text, so you can see exactly what a training/validation row looks like instead of trusting the pipeline blindly.

Each row can contain **several documents** glued together (short documents share a row) or a **piece of one long document** (a long document spans several rows). The `<eos>` token marks every document boundary.

In [2]:
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer

PACKED_ROOT = Path("../../../../dataset/final_packed").resolve()
TOKENIZER_REPO = "AxisQuant/IndicPrayog-tokenizer-64k"
SEQ_LEN = 2048
NUM_EXAMPLES = 3
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO)
rng = np.random.RandomState(SEED)


def load_packed(path):
    tokens = np.memmap(path, dtype=np.uint16, mode="r")
    rows = tokens.reshape(-1, SEQ_LEN)
    return rows


def show_rows(path, num_examples=NUM_EXAMPLES):
    rows = load_packed(path)
    print(f"{path.name}: {rows.shape[0]:,} rows x {SEQ_LEN} tokens = {rows.size:,} tokens")

    example_indices = rng.choice(rows.shape[0], size=min(num_examples, rows.shape[0]), replace=False)
    for row_idx in example_indices:
        row = rows[row_idx].tolist()
        text = tokenizer.decode(row, skip_special_tokens=False)
        documents = [doc.strip() for doc in text.split("<eos>") if doc.strip()]
        print(f"\n--- row {row_idx} ({len(documents)} document piece(s) in this row) ---")
        for doc in documents:
            preview = doc[:300] + ("..." if len(doc) > 300 else "")
            print(f"  > {preview}")
    return rows

#### train.bin : a few random rows

Shard order was shuffled before packing, so a single row can, and often does, mix Hindi, English, and Marathi text.

In [3]:
train_rows = show_rows(PACKED_ROOT / "train.bin")

train.bin: 4,863,624 rows x 2048 tokens = 9,960,701,952 tokens

--- row 4061541 (7 document piece(s) in this row) ---
  > हैं।
  > इस्लामाबाद, (भाषा)। पाकिस्तान सेना ने आज बताया कि पिछले सप्ताह अपने एक अभियान के तहत उसने अशांत बलूचिस्तान प्रांत में इस्लामिक स्टेट के "िकानों को तबाह कर दिया और दो आत्मघाती बम हमलावरों समेत 12 कट्टर आतंकवादियों को मार गिराया है।
 सेना ने तीन दिन के अभियान का विस्तृत ब्योरा देते हुए अपने बयान में क...
  > रायपुर. फाल्गुन मास के कृष्ण पक्ष की एकादशी तिथि को विजया एकादशी का व्रत रखा जाता है. इस दिन शुभ मुहूर्त में व्रत रखते हुए विधि विधान से भगवान विष्णु की पूजा की जाती है. शत्रुओं को परास्त करने के लिए और अपने कार्यों में सफलता प्राप्ति के लिए विजया एकादशी का व्रत किया जाता है.
 फाल्गुन मास के कृष्ण प...
  > इंदौर : भारत में लगातार कलाकारों द्वारा सरकारी पुरस्कार लौटाने के सिलसिले की मशहूर लेखक चेतन भगत ने जमकर आलोचना की और कहा कि आज देश में संकट की ऐसी कोई स्थिति नहीं आई है, जिसकी वजह से सम्मान वापस कर दिए जायें. चेतन भगत ने नवंबर में प्रस्तावित 'इंदौर लिट

#### val_<language>.bin : a few random rows per language

These were held out before tokenization ever ran, the model never trains on these rows.

In [4]:
val_rows = {}
for language in ["en", "hi", "mr"]:
    print(f"\n===== val_{language}.bin =====")
    val_rows[language] = show_rows(PACKED_ROOT / f"val_{language}.bin")


===== val_en.bin =====
val_en.bin: 8,601 rows x 2048 tokens = 17,614,848 tokens

--- row 2654 (1 document piece(s) in this row) ---
  > m making. ” The value of this intense and purposeful interaction between a supervisors and subsidiary should non be underestimated.
Motivation and Satisfaction:
Performance assessment can hold a profound consequence on degrees of employee motive and satisfaction – for better every bit good as for wo...

--- row 2912 (2 document piece(s) in this row) ---
  > force/torque data generated in the virtual environment combined with a priori knowledge about the task will be used to identify and learn the skills in the newly demonstrated tasks and then to reproduce them in the robotics system. The peg-in-hole insertion problem is used as a case study. The overa...
  > 1955 Synod Convention Essay
Respecting the authority of the Bible is distinctively Lutheran. The Roman Catholic Church acknowledges the principle of the authority of the Bible but it violates tha

#### Final token totals

Sanity check against the 10B token target and the `final_pack_report.csv` written by `pack_dataset.ipynb`.

In [5]:
train_tokens = train_rows.size
val_tokens = {language: rows.size for language, rows in val_rows.items()}

print(f"train tokens: {train_tokens:,}")
for language, count in val_tokens.items():
    print(f"val_{language} tokens: {count:,}")
print(f"\ntotal tokens (train + val): {train_tokens + sum(val_tokens.values()):,}  (target: 10,000,000,000)")

train tokens: 9,960,701,952
val_en tokens: 17,614,848
val_hi tokens: 20,254,720
val_mr tokens: 12,746,752

total tokens (train + val): 10,011,318,272  (target: 10,000,000,000)
